# Notebook 11: LSTM Language Model for Text Generation (PyTorch). SOLUTION

ML & NLP course. Data Trainers LLC. Axel Sirota.

## The Scenario

Airbnb has over 200,000 listings, each needing a human-written description. We build a neural language model trained on real Airbnb descriptions that can generate plausible rental text word-by-word.

## Learning Objectives (recap)

1. Autoregressive next-token prediction as the basis for language modeling.
2. LSTM gates and why they solve the vanishing gradient problem.
3. Sliding-window `(x, y)` training pairs with teacher forcing.
4. `Embedding -> LSTM -> Linear` architecture in PyTorch.
5. Training loop with gradient clipping; perplexity tracking.
6. Greedy, temperature, and top-k generation with hidden state carry-over.

## Section 0: Environment Setup

In [ ]:
# Install required packages (Colab)
!pip install -q torch textblob pandas numpy matplotlib
!python -m textblob.download_corpora

In [ ]:
# Imports grouped by purpose (stdlib -> visualization -> data -> model -> training)
import math
import random
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from textblob import TextBlob

warnings.filterwarnings('ignore')

# Hyperparameters (single source of truth)
SEED            = 42
CORPUS_SIZE     = 20_000
VOCAB_MIN_FREQ  = 3
EMBEDDING_DIM   = 128
HIDDEN_DIM      = 256
NUM_LAYERS      = 2
DROPOUT         = 0.3
SEQ_LEN         = 20
BATCH_SIZE      = 64
EPOCHS          = 10
LR              = 1e-3
GRADIENT_CLIP   = 1.0
MAX_NEW_TOKENS  = 40
TEMPERATURE     = 0.8
TOP_K           = 20

# Seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("\nEnvironment setup complete.")

## Section 1: What is a Language Model?

A language model assigns $P(w_t \mid w_1, \ldots, w_{t-1})$. Generation = sample from it repeatedly (autoregressive). Perplexity = `exp(CE loss)`, and lower is better.

In [ ]:
# Demo: perplexity intuition
vocab_size_demo = 5000
initial_loss = math.log(vocab_size_demo)
initial_ppl  = math.exp(initial_loss)

print(f"For vocab_size={vocab_size_demo}:")
print(f"  Expected initial CE loss : {initial_loss:.3f}")
print(f"  Expected initial PPL     : {initial_ppl:.1f}")
print()
print("After training well, a decent LSTM achieves PPL ~100-200 on Airbnb data.")

## Section 2: RNN and LSTM Theory

Vanilla RNNs suffer from vanishing gradients: backpropagation through many time steps multiplies small weights repeatedly, zeroing out gradients. LSTMs solve this with a cell state highway and three learned gates (forget, input, output) that add information rather than multiply it.

In [ ]:
# Demo: walk through nn.LSTM shapes, a common source of confusion
demo_lstm = nn.LSTM(
    input_size=100,
    hidden_size=128,
    num_layers=2,
    batch_first=True   # expect (B, T, D), not (T, B, D)
)

B, T, D_in = 4, 10, 100
fake_input = torch.randn(B, T, D_in)

output, (h_n, c_n) = demo_lstm(fake_input)

print("Input shape             :", fake_input.shape,  "  (B, T, input_size)")
print("Output shape            :", output.shape,      "  (B, T, hidden_size)")
print("h_n shape (hidden state):", h_n.shape,         "  (num_layers, B, hidden_size)")
print("c_n shape (cell state)  :", c_n.shape,         "  (num_layers, B, hidden_size)")
print()
print("batch_first=True: input = (B, T, D), same convention as nn.Linear and nn.Embedding.")
print("Without it: input would need to be (T, B, D), which is less intuitive.")

## Section 3: Data Preparation

Pipeline: load -> tokenize -> build vocab (special tokens: pad/unk/bos/eos) -> encode -> sliding windows -> Dataset + DataLoader.

In [ ]:
# Load Airbnb rental descriptions
TRAIN_URL = 'https://www.dropbox.com/scl/fi/rbrynlq7871cshi0krftj/train_corpus_descriptions_airbnb.csv?rlkey=td1pfjgqjccap0xu9g4eliube&dl=1'
TEST_URL  = 'https://www.dropbox.com/scl/fi/eys05bzwwnhskadqh7aux/test_corpus_descriptions_airbnb.csv?rlkey=p1zuz90khh5t7dx3hkfba1dzm&dl=1'

train_df = pd.read_csv(TRAIN_URL, header=None, names=['description']).dropna()
test_df  = pd.read_csv(TEST_URL,  header=None, names=['description']).dropna()

print(f"Train descriptions : {len(train_df):,}")
print(f"Test descriptions  : {len(test_df):,}")

train_df = train_df.sample(n=min(CORPUS_SIZE, len(train_df)), random_state=SEED).reset_index(drop=True)
test_df  = test_df.sample(n=min(5000, len(test_df)),          random_state=SEED).reset_index(drop=True)

print(f"\nSubsampled train   : {len(train_df):,}")
print(f"Subsampled test    : {len(test_df):,}")
print("\nFirst description:")
print(train_df.iloc[0].description[:300])

In [ ]:
# Tokenize with TextBlob
def tokenize(text):
    """Lowercase tokenization via TextBlob."""
    return [w.lower() for w in TextBlob(str(text)).words]

sample_tokens = tokenize(train_df.iloc[0].description)
print("First 20 tokens:", sample_tokens[:20])

print("\nTokenizing all descriptions...")
train_tokens = [tokenize(t) for t in train_df['description']]
test_tokens  = [tokenize(t) for t in test_df['description']]

print(f"Tokenized {len(train_tokens):,} train descriptions")
lengths = [len(t) for t in train_tokens]
print(f"Avg tokens/description: {np.mean(lengths):.1f}  |  max: {max(lengths)}")

In [ ]:
# Build vocabulary with special tokens
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

def build_vocab(all_token_lists, min_freq=3):
    counts = Counter(tok for tokens in all_token_lists for tok in tokens)
    kept = [w for w, c in counts.most_common() if c >= min_freq]
    word2id = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for w in kept:
        if w not in word2id:
            word2id[w] = len(word2id)
    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word

word2id, id2word = build_vocab(train_tokens, min_freq=VOCAB_MIN_FREQ)
vocab_size = len(word2id)

PAD_IDX = word2id[PAD_TOKEN]   # 0
UNK_IDX = word2id[UNK_TOKEN]   # 1
BOS_IDX = word2id[BOS_TOKEN]   # 2
EOS_IDX = word2id[EOS_TOKEN]   # 3

print(f"Vocabulary size (min_freq={VOCAB_MIN_FREQ}): {vocab_size:,}")
print(f"Special token ids: PAD={PAD_IDX}, UNK={UNK_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")
print(f"Sample vocab entries: {list(word2id.items())[4:14]}")

In [ ]:
# Encode token lists to integer ids with BOS/EOS wrapping
def encode(tokens):
    """Convert token strings to integer ids. Adds BOS at start, EOS at end."""
    ids = [word2id.get(t, UNK_IDX) for t in tokens]
    return [BOS_IDX] + ids + [EOS_IDX]

demo_ids = encode(train_tokens[0])
print("Tokens (first 10):", train_tokens[0][:10])
print("IDs    (first 12):", demo_ids[:12])
print("Decoded back     :", [id2word[i] for i in demo_ids[:12]])

train_ids = [encode(t) for t in train_tokens]
test_ids  = [encode(t) for t in test_tokens]
print(f"\nEncoded {len(train_ids):,} train sequences")

In [ ]:
# Demo: sliding-window pair generation
def make_windows(ids, seq_len=SEQ_LEN):
    """Return list of (x, y) integer-list pairs of length seq_len."""
    pairs = []
    for i in range(len(ids) - seq_len):
        x = ids[i     : i + seq_len]
        y = ids[i + 1 : i + seq_len + 1]
        pairs.append((x, y))
    return pairs

toy_ids   = demo_ids[:8]
toy_words = [id2word[i] for i in toy_ids]
print("Example sequence :", toy_words)
print()
for x, y in make_windows(toy_ids, seq_len=4):
    x_words = [id2word[i] for i in x]
    y_words = [id2word[i] for i in y]
    print(f"  x={x_words}  ->  y={y_words}")

### Lab 1: `LMDataset` (SOLUTION)

Wraps sliding-window pair generation into a PyTorch `Dataset` and `DataLoader`.

In [ ]:
# Solution: LMDataset

class LMDataset(Dataset):
    def __init__(self, encoded_sequences, seq_len=SEQ_LEN):
        # Flatten all (x, y) windows from every sequence into a single list.
        # For 20K descriptions with avg 50 tokens, this yields ~600K training pairs.
        self.pairs = []
        for seq in encoded_sequences:
            self.pairs.extend(make_windows(seq, seq_len))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # Convert stored Python lists to torch.long tensors.
        # DataLoader stacks multiple items along dim 0 to form a batch.
        x, y = self.pairs[idx]
        return (
            torch.tensor(x, dtype=torch.long),   # (SEQ_LEN,)
            torch.tensor(y, dtype=torch.long),   # (SEQ_LEN,)
        )


# Build datasets and DataLoaders
train_dataset = LMDataset(train_ids)
test_dataset  = LMDataset(test_ids)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train pairs: {len(train_dataset):,}")
print(f"Test pairs : {len(test_dataset):,}")
x_batch, y_batch = next(iter(train_loader))
print(f"x_batch shape: {x_batch.shape}  (expected: ({BATCH_SIZE}, {SEQ_LEN}))")
print(f"y_batch shape: {y_batch.shape}  (expected: ({BATCH_SIZE}, {SEQ_LEN}))")
print(f"x dtype: {x_batch.dtype}, y dtype: {y_batch.dtype}")

# Notes:
# - All pairs live in RAM as Python lists. For 20K descriptions this is fine.
#   For very large corpora (millions of docs) you would use an IterableDataset
#   that generates windows on the fly to avoid memory exhaustion.
# - y is x shifted right by 1: if x = [bos, w1, w2, ..., wN], y = [w1, w2, ..., wN+1].
#   The model learns "given wN...w1 context, predict the next token".

## Section 4: The LSTM Language Model

Architecture: `(B,T) ids -> Embedding(V,E) -> LSTM(E,H,L) -> Linear(H,V) -> (B,T,V) logits`.

In [ ]:
# Demo: shape walkthrough
demo_vocab  = 100
demo_emb_l  = nn.Embedding(demo_vocab, EMBEDDING_DIM, padding_idx=0)
demo_lstm_l = nn.LSTM(EMBEDDING_DIM, HIDDEN_DIM, num_layers=NUM_LAYERS,
                      dropout=DROPOUT, batch_first=True)
demo_fc_l   = nn.Linear(HIDDEN_DIM, demo_vocab)

x_demo = torch.randint(0, demo_vocab, (4, SEQ_LEN))
e_demo              = demo_emb_l(x_demo)
out_demo, state_d   = demo_lstm_l(e_demo)
logits_demo         = demo_fc_l(out_demo)

print("x shape    :", x_demo.shape,        "  (B, T)")
print("emb shape  :", e_demo.shape,        "  (B, T, EMBEDDING_DIM)")
print("out shape  :", out_demo.shape,      "  (B, T, HIDDEN_DIM)")
print("h_n shape  :", state_d[0].shape,   "  (num_layers, B, HIDDEN_DIM)")
print("logits     :", logits_demo.shape,   "  (B, T, vocab_size)")

### Lab 2: `LSTMLanguageModel` (SOLUTION)

In [ ]:
# Solution: LSTMLanguageModel

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=NUM_LAYERS, dropout=DROPOUT, pad_idx=PAD_IDX):
        super().__init__()
        # Embedding lookup table.
        # padding_idx=pad_idx pins the PAD embedding to the zero vector and gives
        # it zero gradient, so padding tokens can't mislead the model.
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # Multi-layer LSTM.
        # batch_first=True: input/output shape is (B, T, D), not (T, B, D).
        # dropout is applied between layers, not after the last layer.
        # PyTorch warns if dropout > 0 with num_layers=1, so we guard against it.
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )

        # Output projection: hidden state -> logits over vocabulary.
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, state=None):
        # x: (B, T) integer token ids
        # state: (h, c) tuple or None; when None PyTorch auto-inits to zeros

        # Embedding lookup -> (B, T, emb_dim)
        e = self.emb(x)

        # LSTM forward pass -> output (B, T, hidden_dim) + new state (h, c).
        # We pass state so the caller can carry hidden state across generation steps.
        out, state = self.lstm(e, state)

        # Project each time-step's hidden state to vocabulary logits -> (B, T, V).
        # Do NOT apply softmax here. CrossEntropyLoss applies log_softmax internally
        # for numerical stability. Applying softmax yourself and then passing it to
        # CrossEntropyLoss is a double-softmax bug that silently kills gradients.
        logits = self.fc(out)

        return logits, state


# Instantiate and verify
model = LSTMLanguageModel(vocab_size).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")
print("  Parameter breakdown:")
print(f"    Embedding : {model.emb.weight.numel():>10,}  (vocab_size × emb_dim)")
print(f"    LSTM      : {sum(p.numel() for p in model.lstm.parameters()):>10,}")
print(f"    Linear    : {sum(p.numel() for p in model.fc.parameters()):>10,}")

x_dummy = torch.randint(0, vocab_size, (4, SEQ_LEN), device=device)
logits_dummy, state_dummy = model(x_dummy)
print(f"\nOutput logits shape: {logits_dummy.shape}  (expected: (4, {SEQ_LEN}, {vocab_size}))")
print(f"h_n shape: {state_dummy[0].shape}   c_n shape: {state_dummy[1].shape}")

# Notes:
# - Why return state? At training time we ignore it (reset each batch).
#   At generation time we carry it forward so the model "remembers" the growing
#   sequence without re-processing all previous tokens on every step.
# - The dominant parameter cost is the Embedding + Linear layers, both of size
#   vocab_size x dim. Tying weights (sharing emb.weight with fc.weight) halves
#   this; we skip it for clarity.

## Section 5: Training Loop + Perplexity

Teacher forcing: at training time we feed the true previous tokens, not the model's predictions. Sliding windows make this automatic. Gradient clipping is mandatory for LSTMs; without it gradients can explode to `nan`.

In [ ]:
# Setup: loss function + sanity check
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print("Loss function:", loss_fn)
print()
print(f"Vocab size = {vocab_size}")
print(f"Expected initial loss       ~ ln({vocab_size}) = {math.log(vocab_size):.3f}")
print(f"Expected initial perplexity ~ {math.exp(math.log(vocab_size)):.0f}")

In [ ]:
# Demo: the reshape trick for CrossEntropyLoss
B_d = 8; T_d = SEQ_LEN; V_d = vocab_size
fake_logits  = torch.randn(B_d, T_d, V_d)
fake_targets = torch.randint(0, V_d, (B_d, T_d))

loss_demo = loss_fn(
    fake_logits.reshape(-1, V_d),   # (B*T, V)
    fake_targets.reshape(-1)        # (B*T,)
)
print(f"logits before reshape : {fake_logits.shape} -> after: {fake_logits.reshape(-1, V_d).shape}")
print(f"targets before reshape: {fake_targets.shape} -> after: {fake_targets.reshape(-1).shape}")
print(f"Loss (random logits)  : {loss_demo.item():.4f}  (should be close to ln({V_d}) = {math.log(V_d):.3f})")

### Lab 3: Training Loop (SOLUTION)

In [ ]:
# Solution: full training loop

# Instantiate model, loss, optimizer.
# Re-instantiate model to ensure clean weights if this cell is re-run.
model     = LSTMLanguageModel(vocab_size).to(device)
loss_fn   = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses, val_losses = [], []
train_ppls,   val_ppls   = [], []

for epoch in range(EPOCHS):
    # Training phase
    model.train()    # activates dropout between LSTM layers
    total_train_loss = 0.0
    n_train = 0

    for x, y in train_loader:
        # a. Move tensors to device; every tensor touching the model must live there
        x = x.to(device)
        y = y.to(device)

        # b. Clear gradients from the previous step.
        #    PyTorch accumulates gradients by default. Skipping this is a common
        #    silent bug: gradients pile up across batches.
        optimizer.zero_grad()

        # c. Forward pass; we discard state (reset each window, not BPTT)
        logits, _ = model(x)

        # d. Compute cross-entropy loss using the reshape trick.
        #    logits: (B, T, V) -> (B*T, V)
        #    y:      (B, T)    -> (B*T,)
        #    ignore_index=PAD_IDX means padding positions contribute 0 to the loss.
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))

        # e. Backpropagation; builds .grad on every parameter
        loss.backward()

        # f. CRITICAL: clip gradient norm to GRADIENT_CLIP (1.0).
        #    Without this, LSTM weights can explode to inf and loss becomes nan.
        #    This is the most common reason LSTM training silently fails.
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)

        # g. Apply the gradient update
        optimizer.step()

        total_train_loss += loss.item()
        n_train += 1

    avg_train = total_train_loss / max(n_train, 1)
    train_losses.append(avg_train)
    train_ppls.append(math.exp(avg_train))

    # Evaluation phase
    model.eval()   # deactivates dropout; no-grad context below disables autograd
    total_val_loss = 0.0
    n_val = 0

    with torch.no_grad():    # saves ~30% memory and speeds up eval
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            logits, _ = model(x)
            loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
            total_val_loss += loss.item()
            n_val += 1

    avg_val = total_val_loss / max(n_val, 1)
    val_losses.append(avg_val)
    val_ppls.append(math.exp(avg_val))

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"train loss={avg_train:.4f} ppl={math.exp(avg_train):.1f} | "
          f"val loss={avg_val:.4f} ppl={math.exp(avg_val):.1f}")

# Common mistakes:
#   - Forgetting optimizer.zero_grad(): gradients accumulate, training is unstable.
#   - Forgetting gradient clipping: LSTM explodes to nan.
#   - Forgetting .to(device) on y: CUDA assertion error on loss_fn.
#   - Calling model.eval() without torch.no_grad(): wastes memory but not wrong.
#   - Softmax in model.forward(): double softmax collapses gradients.

In [ ]:
# Plot training and validation curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='train')
axes[0].plot(range(1, len(val_losses)+1),   val_losses,   'r-o', label='val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-entropy Loss'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(train_ppls)+1), train_ppls, 'b-o', label='train')
axes[1].plot(range(1, len(val_ppls)+1),   val_ppls,   'r-o', label='val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('Perplexity (lower is better)'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Final train PPL: {train_ppls[-1]:.1f}  |  Final val PPL: {val_ppls[-1]:.1f}")

In [ ]:
# Save checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'vocab_size': vocab_size,
    'word2id': word2id,
    'id2word': id2word,
    'train_ppls': train_ppls,
    'val_ppls': val_ppls,
}, 'lstm_lm_checkpoint.pt')
print("Checkpoint saved: lstm_lm_checkpoint.pt")

## Section 6: Text Generation

Inference: start from a prompt, warm up the hidden state, then generate one token at a time while carrying state forward. Key strategies: greedy (deterministic), temperature (stochastic), top-k (bounded stochastic).

In [ ]:
# Demo: complete generate() function
@torch.no_grad()
def generate_demo(model, prompt_words, max_new_tokens=MAX_NEW_TOKENS,
                  temperature=TEMPERATURE, top_k=None):
    model.eval()
    prompt_ids = [BOS_IDX] + [word2id.get(w.lower(), UNK_IDX) for w in prompt_words]
    ids = torch.tensor(prompt_ids, device=device).unsqueeze(0)

    logits, state = model(ids, state=None)
    generated = list(prompt_ids)

    for _ in range(max_new_tokens):
        last_logits = logits[:, -1, :]
        if temperature <= 0:
            next_id = last_logits.argmax(dim=-1)
        else:
            scaled = last_logits / temperature
            if top_k is not None and top_k > 0:
                top_vals, top_idx = torch.topk(scaled, top_k, dim=-1)
                probs = torch.zeros_like(scaled).scatter_(1, top_idx, torch.softmax(top_vals, dim=-1))
            else:
                probs = torch.softmax(scaled, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)

        next_id_val = next_id.item()
        generated.append(next_id_val)
        if next_id_val == EOS_IDX:
            break

        next_token = next_id.unsqueeze(0)
        logits, state = model(next_token, state)

    words = []
    for i in generated[1:]:
        if i == EOS_IDX:
            break
        words.append(id2word.get(i, UNK_TOKEN))
    return ' '.join(words)


# Quick test
text = generate_demo(model, ['This', 'rental', 'offers'], temperature=0.8, top_k=20)
print("Generated:", text)

### Lab 4: Implement `generate()` (SOLUTION)

In [ ]:
# Solution: generate()

@torch.no_grad()
def generate(model, prompt_words, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, top_k=TOP_K):
    """Generate text autoregressively using the trained LSTM language model."""
    model.eval()

    # Encode prompt and prepend BOS.
    # UNK_IDX handles out-of-vocab prompt words gracefully.
    prompt_ids = [BOS_IDX] + [word2id.get(w.lower(), UNK_IDX) for w in prompt_words]
    ids = torch.tensor(prompt_ids, device=device).unsqueeze(0)   # (1, T_prompt)

    # Warm up: process the full prompt to initialize the hidden state.
    # The key efficiency insight is that we run the prompt through the model once
    # to get (logits, state). At each subsequent step we only pass the single
    # latest token, and the state carries all prior context.
    logits, state = model(ids, state=None)

    generated = list(prompt_ids)

    for _ in range(max_new_tokens):
        # Extract last-position logits; this is the prediction for the next token.
        last_logits = logits[:, -1, :]   # (1, V)

        if temperature <= 0:
            # Greedy: deterministic, always picks the most likely token.
            # Tends to produce repetitive sequences ("bedroom bedroom bedroom").
            next_id = last_logits.argmax(dim=-1)   # (1,)
        else:
            # Temperature: scale logits before softmax.
            # T<1 gives a sharper distribution (less variety, safer text).
            # T>1 gives a flatter distribution (more variety, may go off-rails).
            scaled = last_logits / temperature      # (1, V)

            if top_k is not None and top_k > 0:
                # Top-k: restrict sampling to the k most likely tokens.
                # This prevents picking from the long tail of unlikely words.
                # Get top-k values and their indices, then scatter normalized
                # probabilities back into a full-vocab tensor.
                top_vals, top_idx = torch.topk(scaled, top_k, dim=-1)   # (1, k) each
                probs = torch.zeros_like(scaled).scatter_(
                    1,
                    top_idx,
                    torch.softmax(top_vals, dim=-1)   # normalize among top-k only
                )  # (1, V): zero everywhere except top-k positions
            else:
                # Standard temperature sampling over the full vocabulary
                probs = torch.softmax(scaled, dim=-1)   # (1, V)

            # Sample one token from the probability distribution.
            # torch.multinomial handles the randomness; seed is set globally.
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)   # (1,)

        next_id_val = next_id.item()
        generated.append(next_id_val)

        # Stop early if EOS is generated
        if next_id_val == EOS_IDX:
            break

        # Carry state forward: pass only the single newly generated token.
        # The naive alternative is to re-run the entire growing sequence through
        # the model at each step (O(n^2) vs O(n)).
        next_token = next_id.unsqueeze(0)           # (1, 1)
        logits, state = model(next_token, state)

    # Decode to words: skip leading BOS, stop at EOS
    words = []
    for i in generated[1:]:
        if i == EOS_IDX:
            break
        words.append(id2word.get(i, UNK_TOKEN))
    return ' '.join(words)


# Verification
prompt = ['This', 'rental', 'offers']
print(f"Prompt: '{' '.join(prompt)}'")
print()
print("Generated (T=0.8, top-k=20):")
print(generate(model, prompt, temperature=0.8, top_k=20))
print()
print("Generated (T=0.0, greedy):")
print(generate(model, prompt, temperature=0.0, top_k=None))
print()
print("Generated (T=1.2, top-k=50):")
print(generate(model, prompt, temperature=1.2, top_k=50))

In [ ]:
# Demo: compare greedy vs. temperature vs. top-k across multiple prompts

prompts = [
    ['This', 'rental', 'offers'],
    ['The', 'apartment', 'is'],
    ['Located', 'in', 'the'],
]

strategies = [
    {'label': 'Greedy (T=0)',         'temperature': 0.0, 'top_k': None},
    {'label': 'Temperature T=0.5',    'temperature': 0.5, 'top_k': None},
    {'label': 'Temperature T=1.0',    'temperature': 1.0, 'top_k': None},
    {'label': 'Top-k=20 T=0.8',       'temperature': 0.8, 'top_k': 20},
]

for prompt in prompts:
    print(f"\nPrompt: '{' '.join(prompt)}'")
    print('-' * 70)
    for s in strategies:
        text = generate(model, prompt, temperature=s['temperature'], top_k=s['top_k'])
        print(f"[{s['label']:25s}]: {text[:80]}")

## Section 7: Ablation and Model Analysis

In [ ]:
# Demo: temperature effect, same prompt with samples at different temperatures

demo_prompt = ['The', 'space', 'features']
temperatures = [0.5, 1.0, 1.5]

print(f"Prompt: '{' '.join(demo_prompt)}'\n")
for t in temperatures:
    samples = [generate(model, demo_prompt, temperature=t, top_k=None, max_new_tokens=30)
               for _ in range(2)]
    print(f"T={t:.1f}:")
    for i, s in enumerate(samples, 1):
        print(f"  Sample {i}: {s}")
    print()

print("Observations:")
print("  T=0.5 -> focused, repetitive, 'safe' (often copies frequent training phrases)")
print("  T=1.0 -> balanced, moderate variety")
print("  T=1.5 -> creative but sometimes grammatically incoherent")

### Lab 5: Tiny model ablation (SOLUTION)

In [ ]:
# Solution: tiny model ablation

# Instantiate small model: hidden_dim=64, num_layers=1
small_model = LSTMLanguageModel(
    vocab_size,
    hidden_dim=64,
    num_layers=1,
    dropout=0.0   # dropout requires num_layers > 1 to apply between layers
).to(device)
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=LR)
small_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

small_n_params = sum(p.numel() for p in small_model.parameters())
full_n_params  = sum(p.numel() for p in model.parameters())
print(f"Small model params: {small_n_params:,}")
print(f"Full  model params: {full_n_params:,}")
print(f"Ratio: {full_n_params / small_n_params:.1f}x bigger")

SMALL_EPOCHS = 3
small_val_ppls = []

# Train for 3 epochs
for epoch in range(SMALL_EPOCHS):
    small_model.train()
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        small_optimizer.zero_grad()
        logits, _ = small_model(x)
        loss = small_loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), GRADIENT_CLIP)
        small_optimizer.step()

    # Evaluate
    small_model.eval()
    total_v, n_v = 0.0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            logits, _ = small_model(x)
            loss = small_loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
            total_v += loss.item()
            n_v += 1
    v_ppl = math.exp(total_v / max(n_v, 1))
    small_val_ppls.append(v_ppl)
    print(f"Small model epoch {epoch+1}/{SMALL_EPOCHS} | val ppl={v_ppl:.1f}")

# Generate samples from small model
print("\nSmall model samples:")
print(" 1:", generate(small_model, ['This', 'rental', 'offers']))
print(" 2:", generate(small_model, ['The', 'apartment', 'has']))

# Compare
full_final_ppl = val_ppls[-1] if val_ppls else float('nan')
print(f"\nSmall model (hidden=64, 1-layer, {SMALL_EPOCHS} epochs) val PPL : {small_val_ppls[-1]:.1f}")
print(f"Full  model (hidden=256, 2-layer, {EPOCHS} epochs) val PPL       : {full_final_ppl:.1f}")
print()
print("Typical result: the full model achieves 30-50% lower perplexity.")
print("The qualitative gap is even larger; full model samples are more coherent.")

# Notes:
#   - More hidden units means more capacity to model the conditional distribution.
#   - A second LSTM layer can learn higher-order sequential patterns.
#   - The improvement is bounded by data size: with only 20K descriptions, very
#     large models won't help much and may overfit.
#   - For production use, the LSTM would be replaced by a GPT-style transformer
#     which can handle much longer context ranges.

### Optional Lab: Generation strategy sweep (SOLUTION)

In [ ]:
# Solution: generation strategy sweep

opt_prompts = [
    ['Enjoy', 'your', 'stay'],
    ['The', 'bedroom', 'has'],
    ['Guests', 'have', 'access'],
]

opt_strategies = [
    {'label': 'Greedy',          'temperature': 0.0, 'top_k': None},
    {'label': 'T=0.8, top-k=20', 'temperature': 0.8, 'top_k': 20},
    {'label': 'T=1.2, top-k=50', 'temperature': 1.2, 'top_k': 50},
]

for prompt in opt_prompts:
    print(f"\nPrompt: '{' '.join(prompt)}'")
    print('=' * 70)
    for s in opt_strategies:
        # Generate 2 samples per strategy to show variance
        for trial in range(1, 3):
            text = generate(model, prompt, temperature=s['temperature'],
                            top_k=s['top_k'], max_new_tokens=30)
            print(f"[{s['label']:15s}] #{trial}: {text[:70]}")

# Discussion:
#   - Greedy: most repetitive across trials (identical for the same prompt/seed).
#     Good for replicability, poor for variety.
#   - T=0.8 + top-k=20: best balance. Rejects the low-probability tail while
#     still sampling from plausible continuations. This is a reasonable default
#     for most production generation systems.
#   - T=1.2 + top-k=50: most diverse but includes occasional grammatical errors
#     or topic drift. Useful for brainstorming, not for polished output.

## Self-check quiz answers

1. Initial loss for `vocab_size=5000`?

Approximately `ln(5000) ~ 8.52`. A random model distributes probability uniformly over all V classes, so cross-entropy = `ln(V)`. If your initial loss is much higher, check for softmax in `forward()`, incorrect vocab size, or a logits/targets shape mismatch.

2. Generation repeats "bedroom bedroom bedroom". What's wrong?

Greedy decoding is getting stuck in a local mode. The word "bedroom" is likely the argmax at every step because the model learned it's the most common word in this context. Fix: use temperature sampling (`T >= 0.7`) and/or top-k sampling to break out of the loop.

3. Why does `ignore_index=PAD_IDX` matter?

Padding positions are added to make sequences uniform in length for batching; they don't represent real tokens. Without `ignore_index`, the model tries to predict `<pad>` at those positions, which would:

- Inflate the denominator (making the average loss appear smaller than it really is).
- Send gradients through the model for positions with no meaningful signal, adding noise.

4. Ten epochs, perplexity still 800. Three likely bugs:

1. Softmax in `forward()`: double softmax collapses gradients silently. Check your model output.
2. Forgot gradient clipping: exploding gradients corrupted weights early in training and they never recovered.
3. Learning rate too high/low: `LR=1e-3` is a good default; `LR=1e-5` would barely move; `LR=1e-1` would diverge.
4. (Bonus) `CrossEntropyLoss` receiving softmax-ed inputs (same as #1 from a different angle).
5. (Bonus) Targets are shifted wrong: `x` and `y` are identical (same window, no offset).

## What you learned

- Autoregressive language modeling: frame generation as repeated next-token prediction.
- Perplexity: `exp(CE)`, the canonical LM metric.
- LSTM gating: forget/input/output gates plus cell state highway solve the vanishing gradient problem.
- Teacher forcing: feed true tokens at train time; feed the model's own output at inference.
- Gradient clipping: mandatory guard against LSTM gradient explosion.
- Hidden state carry-over: efficient O(n) generation that avoids reprocessing history.
- Temperature and top-k: the creativity/coherence trade-off knobs.

## Limitations and next steps

LSTMs have a fundamental ceiling: the hidden state must compress all prior context into a fixed-size vector. Beyond ~50 tokens the early context is effectively lost.

Transformers fix this. Attention looks directly at every previous token (no compression) and the computation is fully parallel over the time dimension. That's why every modern LLM is transformer-based.

Next up, Notebook 12: fine-tune `gpt2` (124M params) on the same Airbnb dataset. Direct comparison of perplexity, coherence, training time, and model size. GPT-2 tends to win on quality but uses ~40x more parameters; the LSTM still does well in low-resource settings.